<a href="https://colab.research.google.com/github/velchan15/Classifying_X-Ray_Images_using_PyTorch/blob/main/Classifying_X-Ray_Images_using_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Pneumonia is one of the leading respiratory illnesses worldwide, and its timely and accurate diagnosis is essential for effective treatment. Manually reviewing chest X-rays is a critical step in this process, and AI can provide valuable support by helping to expedite the assessment. In your role as a consultant data scientist, you will test the ability of a deep learning model to distinguish pneumonia cases from normal images of lungs in chest X-rays.

By fine-tuning a pre-trained convolutional neural network, specifically the ResNet-18 model, your task is to classify X-ray images into two categories: normal lungs and those affected by pneumonia. You can leverage its already trained weights and get an accurate classifier trained faster and with fewer resources.

## The Data

<img src="x-rays_sample.png" align="center"/>
&nbsp

You have a dataset of chest X-rays that have been preprocessed for use with a ResNet-18 model. You can see a sample of 5 images from each category above. Upon unzipping the `chestxrays.zip` file (code provided below), you will find your dataset inside the `data/chestxrays` folder divided into `test` and `train` folders.

There are 150 training images and 50 testing images for each category, NORMAL and PNEUMONIA (300 and 100 in total). For your convenience, this data has already been loaded into a `train_loader` and a `test_loader` using the `DataLoader` class from the PyTorch library.

In [3]:
!pip install torchmetrics
# Import required libraries
# -------------------------
# Data loading
import random
import numpy as np
from torchvision.transforms import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# Train model
import torch
from torchvision import models
import torch.nn as nn
import torch.optim as optim

# Evaluate model
from torchmetrics import Accuracy, F1Score

# Set random seeds for reproducibility
torch.manual_seed(101010)
np.random.seed(101010)
random.seed(101010)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 15.8 MB/s eta 0:00:00


In [12]:
import os
import zipfile

# Ensure the 'data' directory exists
os.makedirs('data', exist_ok=True)

# Define the expected path for the zip file
zip_path = 'data/chestxrays.zip'

# Check if the zip file exists in the root directory and move it if found
if not os.path.exists(zip_path) and os.path.exists('chestxrays.zip'):
    print("Moving 'chestxrays.zip' from root to 'data/' directory...")
    os.rename('chestxrays.zip', zip_path)

# Check if the chestxrays.zip file exists in the data directory
if not os.path.exists(zip_path):
    print(f"Error: '{zip_path}' not found.\n")
    print("Please upload 'chestxrays.zip' to the 'data/' folder using the Colab file browser, ")
    print("or uncomment and replace 'YOUR_DOWNLOAD_LINK_TO_CHESTRAYS.ZIP' with a direct download link if available:")
    print("# !wget -P data/ YOUR_DOWNLOAD_LINK_TO_CHESTRAYS.ZIP")
    # raise FileNotFoundError(f"'{zip_path}' not found. Please upload or download the file.")

# Unzip the data folder if it hasn't been unzipped yet
# This step will only run if the 'data/chestxrays' directory does not exist
if not os.path.exists('data/chestxrays'):
    try:
        print(f"Attempting to unzip '{zip_path}'...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('data')
        print("chestxrays.zip extracted successfully.")
    except FileNotFoundError:
        print(f"Error: '{zip_path}' still not found. Cannot extract. Please ensure the file is in the correct location.")
    except zipfile.BadZipFile:
        print(f"Error: '{zip_path}' is a bad zip file. Please check the integrity of the downloaded/uploaded file.")
    except Exception as e:
        print(f"An unexpected error occurred during unzipping: {e}")

In [13]:
# Define the transformations to apply to the images for use with ResNet-18
transform_mean = [0.485, 0.456, 0.406]
transform_std =[0.229, 0.224, 0.225]
transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize(mean=transform_mean, std=transform_std)])

# Apply the image transforms
train_dataset = ImageFolder('data/chestxrays/train', transform=transform)
test_dataset = ImageFolder('data/chestxrays/test', transform=transform)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=len(train_dataset) // 2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset))

### Below is the provided model evaluation code. Run the below cell to help you evaluate the accuracy and F1-score of your fine-tuned model.

In [14]:
# 1. Load model pre-trained ResNet-18 ke variabel resnet18
resnet18 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 2. Freeze semua bobot layer terdahulu
for param in resnet18.parameters():
    param.requires_grad = False

# 3. Ubah output layer terakhir (fc) menjadi 1 sesuai standar binary DataCamp
num_features = resnet18.fc.in_features
resnet18.fc = nn.Linear(num_features, 1) # <--- DIUBAH JADI 1

# 4. Definisikan device, loss, dan optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet18 = resnet18.to(device)

# Ganti loss function menjadi BCEWithLogitsLoss (khusus untuk 1 output biner)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(resnet18.fc.parameters(), lr=0.001)

# 5. Training Loop selama 3 Epoch
epochs = 3
resnet18.train()

for epoch in range(epochs):
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Karena BCEWithLogitsLoss meminta label dalam bentuk float dan dimensi yang sama dengan output [batch_size, 1]
        labels = labels.unsqueeze(1).float()

        optimizer.zero_grad()
        outputs = resnet18(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 129MB/s]


Epoch 1/3 - Loss: 0.6941
Epoch 2/3 - Loss: 0.6431
Epoch 3/3 - Loss: 0.6169


In [15]:
# Set model to evaluation mode
model = resnet18
model.eval()

# Initialize metrics for accuracy and F1 score
accuracy_metric = Accuracy(task="binary")
f1_metric = F1Score(task="binary")

# Create lists to store all predictions and labels
all_preds = []
all_labels = []

# Disable gradient calculation for evaluation
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)

        # Forward pass
        outputs = model(inputs)

        # Karena output-nya sekarang sudah 1 layer, langsung kenakan sigmoid dan round
        preds = torch.sigmoid(outputs).round()  # Menghasilkan 0 atau 1

        # Kumpulkan semua prediksi dan label ke dalam list
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.unsqueeze(1).tolist())

# Convert lists berkala di luar loop agar tidak error .extend()
all_preds_tensor = torch.tensor(all_preds)
all_labels_tensor = torch.tensor(all_labels)

# Hitung akurasi dan F1 Score
test_accuracy = accuracy_metric(all_preds_tensor, all_labels_tensor).item()
test_f1_score = f1_metric(all_preds_tensor, all_labels_tensor).item()

# Cetak hasil dengan format 3 desimal
print(f"test_accuracy: {test_accuracy:.3f}")
print(f"test_f1_score: {test_f1_score:.3f}")

test_accuracy: 0.550
test_f1_score: 0.571


# Analisis X-Ray Dada untuk Deteksi Pneumonia

Proyek ini bertujuan untuk mengembangkan model *deep learning* yang mampu mengklasifikasikan citra X-ray dada sebagai 'Normal' atau 'Pneumonia'. Diagnosa dini pneumonia sangat krusial, dan model ini diharapkan dapat membantu mempercepat proses penilaian oleh ahli medis.

## Metodologi

Kami menggunakan pendekatan *transfer learning* dengan melakukan *fine-tuning* pada model **ResNet-18** yang sudah dilatih sebelumnya (pretrained) pada dataset ImageNet. Langkah-langkah utama yang dilakukan adalah sebagai berikut:

1.  **Inisialisasi Model**: Model ResNet-18 dimuat dengan bobot default yang sudah dilatih.
2.  **Pembekuan Layer**: Semua *layer* kecuali *final fully connected layer* (output layer) dibekukan (`param.requires_grad = False`) untuk mempertahankan fitur yang sudah dipelajari dan mengurangi waktu pelatihan.
3.  **Modifikasi Output Layer**: *Final fully connected layer* (`fc`) dimodifikasi agar menghasilkan 1 *output*, sesuai dengan kebutuhan klasifikasi biner (Pneumonia vs. Normal).
4.  **Fungsi Loss dan Optimizer**: Digunakan `nn.BCEWithLogitsLoss()` sebagai fungsi *loss* yang sesuai untuk klasifikasi biner dengan *output* tunggal, dan `optim.Adam` sebagai *optimizer* dengan *learning rate* `0.001`.
5.  **Pelatihan Model**: Model dilatih selama **3 epoch** menggunakan *training dataset*.

## Data

Dataset terdiri dari citra X-ray dada yang terbagi menjadi folder `train` dan `test` (`data/chestxrays`). Dataset ini sudah diproses dan dimuat menggunakan `ImageFolder` dan `DataLoader` dari PyTorch, dengan transformasi standar (normalisasi) yang cocok untuk ResNet-18.

## Hasil Evaluasi

Setelah pelatihan, model dievaluasi menggunakan *testing dataset* untuk mengukur performa klasifikasinya. Metrik yang digunakan adalah Akurasi dan F1-Score untuk klasifikasi biner.

Hasil evaluasi model adalah sebagai berikut:

*   **Akurasi Uji (test_accuracy)**: `0.550`
*   **F1-Score Uji (test_f1_score)**: `0.571`

### Interpretasi Singkat:

Hasil akurasi sekitar 55% dan F1-Score sekitar 57% menunjukkan bahwa model ini memiliki performa yang sedikit lebih baik dari menebak secara acak (50%), namun masih perlu banyak peningkatan untuk menjadi alat diagnostik yang handal. Ini bisa disebabkan oleh beberapa faktor seperti:

*   **Ukuran Dataset**: Dataset yang relatif kecil (300 citra pelatihan total) mungkin membatasi kemampuan model untuk belajar fitur yang lebih kompleks.
*   **Jumlah Epoch**: Pelatihan hanya selama 3 *epoch* mungkin tidak cukup bagi model untuk sepenuhnya berkonvergen.
*   **Konfigurasi Model**: Mungkin diperlukan penyesuaian lebih lanjut pada arsitektur model atau *hyperparameter* pelatihan.